Task 4: Heatwaves
The next task is to look for heatwaves in Ås in 2025
In Norway a heatwave is defined as a consecutive period of five or more day where
the max temperature is >= 27 degrees Celcius.
Note that you need max temperature here, not mean. Max temperature is coded as max(air_temperature P1D).
met.no article about heatwaves (only in Norwegian)
Write a function that count the number of heatwaves in a year.
The function should write a summary to a yml file, formatted as the example file
Write the function so that it will work with any station name/id and year

In [14]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from frost_id import client_id
import yaml

client_id = client_id

#Requests the daily mean air temperture for all of 2025
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
'sources': 'SN17850',
'elements': 'max(air_temperature P1D)',
'referencetime': '2025-01-01/2025-12-31',
}

# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id,''))
# Extract JSON data
data = r.json()



#Function that takes a station_id and year as input and returns a list of dates of when it was a heatwave
def heatwaves(station_id, year):
    hot_days = []
    print(f"Amount of observations in data: {len(data['data'])}")
    for observation in data['data']:
        if station_id in observation['sourceId'] and str(year) in observation['referenceTime']:
            if observation['observations'][0]['value'] >= 27:
                hot_days.append(observation['referenceTime'])
    
    #Sjekker om dagene er etter hverandre
    def find_heatwaves(hot_days):
        if not hot_days:
            return []
            
        # Gjør om til datoer, sorter og fjern like datoer (hvis det er flere målinger samme dag)
        dates = pd.to_datetime(hot_days).sort_values().unique()
        
        heatwave_periods = []
        current_streak = []
        
        for dato in dates:
            # Hvis listen er tom, eller datoen er dagen etter den forrige
            if not current_streak or (pd.Timestamp(dato) - pd.Timestamp(current_streak[-1])).days == 1:
                current_streak.append(dato)
            else:
                # Hvis rekken ble brutt, sjekker om den var lang nok (5 eller flere dager)
                if len(current_streak) >= 5:
                    start_date = current_streak[0].strftime("%Y-%m-%d")
                    end_date = current_streak[-1].strftime("%Y-%m-%d")
                    heatwave_periods.append(f"Fra {start_date} til {end_date}")
                
                # Starter en ny rekke med denne datoen
                current_streak = [dato]
                
        # Sjekker den siste rekken når loopen er ferdig
        if len(current_streak) >= 5:
            start_date = current_streak[0].strftime("%Y-%m-%d")
            end_date = current_streak[-1].strftime("%Y-%m-%d")
            heatwave_periods.append(f"Fra {start_date} til {end_date}")
            
        return heatwave_periods
    result = find_heatwaves(hot_days)
    return result


heatwave_days_2025 = heatwaves('SN17850', 2025)
print(heatwave_days_2025)


#Creates a summary of the heatwaves in the form of a yaml file

#Creates a dictionary first (is more practical prior to crateing the file)
Summary = {
    "title": f"Summary of heatwaves in 2025",
    "station_id": "SN17850",
    "year": 2025,
    "heatwave_dates": heatwave_days_2025  # List of dates
}

#Creates the yaml file and writes the summary in it
with open('heatwave_report.yaml', 'w') as file:
    yaml.dump(Summary, file, default_flow_style=False)



Amount of observations in data: 364
['Fra 2025-07-16 til 2025-07-20']
